# DriveGuard — End-to-End Pipeline Evaluation

Evaluates the **full** two-stage model on the labeled test set:

```
16 raw JPEG frames (384×384 ROI crop)
         ↓
  ViT-SO400M  (spatial — classifier head removed)
         ↓  (16, 1152) features
  SingleViewDriveTransformer  (temporal)
         ↓
  Safe / Drink / Phone
```

**No pre-cached `.npy` features** — the ViT runs live on the test frames, same as `infer.py`.

---
### Outputs
- Per-class precision / recall / F1 (`classification_report`)
- Confusion matrix heatmap → saved to Drive
- Overall accuracy + macro F1

In [ ]:
# Cell 1 — Environment Setup, Drive Mount, Device & AMP
import os, gc, glob, json, shutil, zipfile, contextlib
import subprocess
import numpy as np
import torch
import torch.nn as nn

subprocess.run(['pip', 'install', '-q', 'torchmetrics', 'seaborn', 'timm'], check=True)

from google.colab import drive
try:
    drive.mount('/content/drive')
except Exception:
    print('Drive already mounted — continuing.')

# ── Paths (edit DRIVE_BASE if your folder is named differently) ───────────────
DRIVE_BASE = '/content/drive/MyDrive/DriveGuard'
OUTPUT_DIR = f'{DRIVE_BASE}/eval_pipeline'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
device     = 'cuda' if torch.cuda.is_available() else 'cpu'
PIN_MEMORY = (device == 'cuda')
print(f'Device : {device}')
if device == 'cuda':
    print(f'  GPU  : {torch.cuda.get_device_name(0)}')
    print(f'  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.benchmark        = True

# ── AMP dtype ─────────────────────────────────────────────────────────────────
if device == 'cuda' and torch.cuda.is_bf16_supported():
    _AMP_DTYPE = torch.bfloat16
    print('  AMP  : bfloat16 (A100)')
elif device == 'cuda':
    _AMP_DTYPE = torch.float16
    print('  AMP  : float16')
else:
    _AMP_DTYPE = None
    print('  AMP  : disabled (CPU)')

def autocast():
    if _AMP_DTYPE is not None:
        return torch.amp.autocast(device_type='cuda', dtype=_AMP_DTYPE)
    return contextlib.nullcontext()

def cuda_cleanup():
    gc.collect()
    if device == 'cuda':
        torch.cuda.empty_cache()

In [ ]:
# Cell 2 — Copy weights & extract dataset from Drive to local disk
# (Drive FUSE is slow for large files — copy once, run fast)
# All paths confirmed from ablaton_temporal_single_cam_stage4_driveguard.ipynb:
#   DRIVE_BASE    = '/content/drive/MyDrive/DriveGuard'
#   OUTPUT_DIR    = f'{DRIVE_BASE}/stage4_single_cam'
#   checkpoint    = 'best_stage4_single_cam_model.pth'  (copied to OUTPUT_DIR at end of training)
#   cfg           = 'cfg_stage4_single_cam.json'        (saved to OUTPUT_DIR at end of training)
# Spatial weights from photo_to_tensor_transformers.ipynb:
#   WEIGHTS_FILENAME = 'best_model_fused_internViT.pth'  (in DRIVE_BASE root)

# ── Paths on Drive ────────────────────────────────────────────────────────────
DRIVE_SPATIAL_WEIGHTS  = f'{DRIVE_BASE}/best_model_fused_internViT.pth'
DRIVE_TEMPORAL_WEIGHTS = f'{DRIVE_BASE}/stage4_single_cam/best_stage4_single_cam_model.pth'
DRIVE_TEMPORAL_CFG     = f'{DRIVE_BASE}/stage4_single_cam/cfg_stage4_single_cam.json'
DRIVE_DATASET_ZIP      = f'{DRIVE_BASE}/ds_driveguard_16frames_roi.nosync.zip'

# ── Local destinations ────────────────────────────────────────────────────────
LOCAL_SPATIAL_WEIGHTS  = '/content/best_model_fused_internViT.pth'
LOCAL_TEMPORAL_WEIGHTS = '/content/best_stage4_single_cam_model.pth'
LOCAL_TEMPORAL_CFG     = '/content/cfg_stage4_single_cam.json'
LOCAL_ZIP              = '/content/ds_driveguard.zip'
DATASET_PATH           = '/content/ds_driveguard_16frames_roi.nosync'

# ── Copy weights & cfg ────────────────────────────────────────────────────────
for src, dst, name in [
    (DRIVE_SPATIAL_WEIGHTS,  LOCAL_SPATIAL_WEIGHTS,  'spatial weights  (1.6 GB)'),
    (DRIVE_TEMPORAL_WEIGHTS, LOCAL_TEMPORAL_WEIGHTS, 'temporal weights (~50 MB)'),
    (DRIVE_TEMPORAL_CFG,     LOCAL_TEMPORAL_CFG,     'temporal cfg'),
]:
    if not os.path.exists(dst):
        print(f'Copying {name} ...')
        shutil.copy(src, dst)
        print(f'  Done: {os.path.getsize(dst)/1e6:.1f} MB')
    else:
        print(f'{name} already local — skipping.')

# ── Extract dataset zip ───────────────────────────────────────────────────────
if not os.path.exists(DATASET_PATH):
    if not os.path.exists(LOCAL_ZIP):
        print('Copying dataset zip from Drive ...')
        shutil.copy(DRIVE_DATASET_ZIP, LOCAL_ZIP)
        print(f'  Done: {os.path.getsize(LOCAL_ZIP)/1e9:.2f} GB')
    print('Extracting dataset ...')
    with zipfile.ZipFile(LOCAL_ZIP, 'r') as z:
        z.extractall('/content/')
    print(f'  Extracted to {DATASET_PATH}')
else:
    print(f'Dataset already extracted at {DATASET_PATH} — skipping.')

# Verify test split exists
test_path = os.path.join(DATASET_PATH, 'test')
assert os.path.isdir(test_path), f'Test split not found at {test_path}'
print(f'\nTest split ready: {test_path}')

In [ ]:
# Cell 3 — CFG (single source of truth — must match training)
CFG = {
    # ── Data
    'classes'          : ['Drink', 'Phone', 'Safe'],
    'num_classes'      : 3,
    'camera'           : 'a_column_co_driver',   # filter: only this camera's clips
    'num_frames'       : 16,
    'img_size'         : 384,
    'norm_mean'        : [0.5, 0.5, 0.5],        # SigLIP — matches feature extraction
    'norm_std'         : [0.5, 0.5, 0.5],

    # ── Temporal architecture (must match cfg_stage4_single_cam.json)
    'input_dim'        : 1152,
    'hidden_dim'       : 512,
    'num_heads'        : 8,
    'num_layers'       : 4,
    'dim_feedforward'  : 2048,
    'dropout'          : 0.3,
    'noise_std'        : 0.0,    # 0.0 at eval time (training used 0.075)

    # ── Paths
    'spatial_weights'  : LOCAL_SPATIAL_WEIGHTS,
    'temporal_weights' : LOCAL_TEMPORAL_WEIGHTS,
    'test_data_root'   : os.path.join(DATASET_PATH, 'test'),
    'output_dir'       : OUTPUT_DIR,

    # ── Eval
    'batch_size'       : 16,  # clips per batch → 16×16 = 256 ViT forward passes per step
                              # reduce to 8 if OOM
}

print('CFG loaded.')
print(f'  Camera     : {CFG["camera"]}')
print(f'  Test root  : {CFG["test_data_root"]}')
print(f'  Output dir : {CFG["output_dir"]}')
print(f'  Batch size : {CFG["batch_size"]} clips ({CFG["batch_size"] * CFG["num_frames"]} ViT calls/batch)')

In [ ]:
# Cell 4 — Dataset
from collections import Counter
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

class EndToEndTestDataset(Dataset):
    """
    Loads 16-frame JPEG sequences from test/{class}/{seq_id}/frame_XX.jpg.
    Filters to sequences whose seq_id starts with cfg['camera'].

    Returns:
        frames : (16, 3, 384, 384) float32 tensor
        label  : scalar long tensor (class index)
    """
    def __init__(self, test_root: str, cfg: dict, transform):
        self.transform = transform
        self.samples   = []   # [(label_idx, [frame_path, ...])])
        cam_prefix     = cfg['camera']
        T              = cfg['num_frames']

        for label_idx, cls in enumerate(cfg['classes']):
            cls_dir = os.path.join(test_root, cls)
            if not os.path.isdir(cls_dir):
                print(f'  [WARNING] Missing: {cls_dir}')
                continue
            for seq_id in sorted(os.listdir(cls_dir)):
                if not seq_id.startswith(cam_prefix):
                    continue
                seq_dir = os.path.join(cls_dir, seq_id)
                frames  = sorted(glob.glob(os.path.join(seq_dir, '*.jpg')))[:T]
                if len(frames) == T:
                    self.samples.append((label_idx, frames))

        counts = Counter(label for label, _ in self.samples)
        print(f'[EndToEndTestDataset]  camera={cam_prefix}')
        for i, cls in enumerate(cfg['classes']):
            print(f'  {cls:8s}: {counts.get(i, 0):4d} sequences')
        print(f'  TOTAL   : {len(self.samples):4d}')

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        label, frame_paths = self.samples[idx]
        imgs = [self.transform(Image.open(p).convert('RGB')) for p in frame_paths]
        return torch.stack(imgs), torch.tensor(label, dtype=torch.long)

In [ ]:
# Cell 5 — Model Architecture (copied verbatim from training notebooks)
import timm

class TemporalEncoder(nn.Module):
    """
    Input:  (B, T=16, D=1152)
    Output: (B, hidden_dim)  via CLS token  (Pre-LN, 4 layers)
    """
    def __init__(self, cfg):
        super().__init__()
        D, H = cfg['input_dim'], cfg['hidden_dim']
        T    = cfg['num_frames']

        self.input_proj = nn.Linear(D, H)
        self.cls_token  = nn.Parameter(torch.zeros(1, 1, H))
        self.pos_embed  = nn.Parameter(torch.zeros(1, T + 1, H))

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=H, nhead=cfg['num_heads'],
            dim_feedforward=cfg['dim_feedforward'],
            dropout=cfg['dropout'], batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=cfg['num_layers'])
        self.norm = nn.LayerNorm(H)

        nn.init.trunc_normal_(self.cls_token, std=0.02)
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x):
        B   = x.size(0)
        x   = self.input_proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x   = torch.cat([cls, x], dim=1) + self.pos_embed
        x   = self.transformer(x)
        return self.norm(x[:, 0])


class SingleViewDriveTransformer(nn.Module):
    """
    TemporalEncoder + Dropout + Linear classifier.
    Input:  (B, T=16, D=1152)
    Output: (B, num_classes) logits
    """
    def __init__(self, cfg):
        super().__init__()
        self.encoder    = TemporalEncoder(cfg)
        self.dropout    = nn.Dropout(cfg['dropout'])
        self.classifier = nn.Linear(cfg['hidden_dim'], cfg['num_classes'])
        self.noise_std  = cfg['noise_std']

    def forward(self, x):
        if self.training and self.noise_std > 0:
            x = x + torch.randn_like(x) * self.noise_std
        h = self.encoder(x)
        return self.classifier(self.dropout(h))

In [ ]:
# Cell 6 — Load Both Models

# ── Spatial: ViT-SO400M with classifier head removed ─────────────────────────
print('Loading spatial model (ViT-SO400M) ...')
spatial_model = timm.create_model('vit_so400m_patch14_siglip_384', pretrained=False, num_classes=3)
spatial_model.load_state_dict(torch.load(CFG['spatial_weights'], map_location='cpu'))
spatial_model.reset_classifier(0)   # strip 3-class head → exposes 1152-dim embedding
if _AMP_DTYPE is not None:
    spatial_model = spatial_model.to(device).to(_AMP_DTYPE).eval()
else:
    spatial_model = spatial_model.to(device).eval()
try:
    spatial_model = torch.compile(spatial_model)
    print('  torch.compile: enabled')
except Exception as e:
    print(f'  torch.compile: skipped ({e})')
print(f'  Parameters : {sum(p.numel() for p in spatial_model.parameters())/1e6:.1f} M')

# ── Temporal: load architecture from saved cfg (guarantees match with weights) ─
print('\nLoading temporal model (SingleViewDriveTransformer) ...')
# Load cfg saved by training notebook to /content/drive/MyDrive/DriveGuard/stage4_single_cam/
t_cfg = json.load(open(LOCAL_TEMPORAL_CFG))
t_cfg['noise_std'] = 0.0   # disable input noise at eval time (training used 0.075)

temporal_model = SingleViewDriveTransformer(t_cfg)
state = torch.load(CFG['temporal_weights'], map_location='cpu')
state = {k.replace('_orig_mod.', ''): v for k, v in state.items()}  # strip torch.compile prefix
temporal_model.load_state_dict(state)
if _AMP_DTYPE is not None:
    temporal_model = temporal_model.to(device).to(_AMP_DTYPE).eval()
else:
    temporal_model = temporal_model.to(device).eval()
try:
    temporal_model = torch.compile(temporal_model)
    print('  torch.compile: enabled')
except Exception as e:
    print(f'  torch.compile: skipped ({e})')
print(f'  Parameters : {sum(p.numel() for p in temporal_model.parameters())/1e6:.1f} M')
print(f'  camera     : {t_cfg["camera"]}')

print('\nBoth models ready.')

In [ ]:
# Cell 7 — Dataset & DataLoader

eval_transform = transforms.Compose([
    transforms.Resize((CFG['img_size'], CFG['img_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=CFG['norm_mean'], std=CFG['norm_std']),
])

test_ds = EndToEndTestDataset(
    test_root = CFG['test_data_root'],
    cfg       = CFG,
    transform = eval_transform,
)

test_loader = DataLoader(
    test_ds,
    batch_size  = CFG['batch_size'],
    shuffle     = False,
    num_workers = 4,   # Colab uses fork by default — safe without __main__ guard
    pin_memory  = PIN_MEMORY,
)

print(f'\nDataLoader ready — {len(test_ds)} sequences, {len(test_loader)} batches')

In [ ]:
# Cell 8 — End-to-End Inference
# Frames → ViT (spatial) → (B, 16, 1152) → Temporal Transformer → predictions

all_preds, all_labels = [], []

spatial_model.eval()
temporal_model.eval()

with torch.no_grad():
    for frames, labels in tqdm(test_loader, desc='End-to-End Eval'):
        # frames : (B, 16, 3, 384, 384)
        B, T, C, H, W = frames.shape

        # Flatten 16 frames per clip into one big batch for the ViT
        flat = frames.view(B * T, C, H, W)
        if _AMP_DTYPE is not None:
            flat = flat.to(device, dtype=_AMP_DTYPE)
        else:
            flat = flat.to(device)

        # Spatial: (B*16, 3, 384, 384) → (B*16, 1152)
        with autocast():
            feats = spatial_model(flat)

        # Reshape back to sequence format: (B, 16, 1152)
        feats = feats.view(B, T, -1)

        # Temporal: (B, 16, 1152) → (B, 3) logits
        with autocast():
            logits = temporal_model(feats)

        preds = logits.float().argmax(dim=1)
        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.tolist())

print(f'\nInference complete — {len(all_preds)} sequences evaluated.')

In [ ]:
# Cell 9 — Metrics, Confusion Matrix & Save
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print('=' * 65)
print('END-TO-END TEST SET EVALUATION')
print('=' * 65)

# ── Classification report ─────────────────────────────────────────────────────
print('\nClassification Report:')
report = classification_report(all_labels, all_preds, target_names=CFG['classes'])
print(report)

# ── Macro F1 + accuracy (sklearn — no version dependency) ────────────────────
macro_f1 = f1_score(all_labels, all_preds, average='macro')
accuracy  = sum(p == l for p, l in zip(all_preds, all_labels)) / len(all_labels)
print(f'Overall Accuracy : {accuracy * 100:.2f}%')
print(f'Macro F1         : {macro_f1:.4f}')

# ── Confusion matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CFG['classes'], yticklabels=CFG['classes'], ax=ax
)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'DriveGuard End-to-End — Test Confusion Matrix\n'
             f'(ViT-SO400M + SingleViewDriveTransformer | {CFG["camera"]})')
plt.tight_layout()

cm_path = os.path.join(CFG['output_dir'], 'confusion_matrix_e2e.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'\nConfusion matrix saved → {cm_path}')

# ── Save classification report as text ───────────────────────────────────────
report_path = os.path.join(CFG['output_dir'], 'classification_report_e2e.txt')
with open(report_path, 'w') as f:
    f.write(f'Overall Accuracy : {accuracy * 100:.2f}%\n')
    f.write(f'Macro F1         : {macro_f1:.4f}\n\n')
    f.write(report)
print(f'Report saved      → {report_path}')

cuda_cleanup()